In [25]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [26]:
TARGET = 'Irrigation_Need'
CAT_COLS = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season',
    'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region'
]

In [27]:
train = pd.read_csv('../data/train.csv')
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [28]:
test = pd.read_csv('../data/test.csv')
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [29]:
X = train.drop(columns=[TARGET, 'id'], errors='ignore').copy()
y = train[TARGET].copy()

label_map = {'Low': 0, 'Medium': 1, 'High': 2}
y = train[TARGET].map(label_map)

# get columns ready for modeling - label encode categoricals, ensuring consistent encoding between train and test
for col in CAT_COLS:
    if col in X.columns:
        le = LabelEncoder()
        if col in test.columns:
            combined = pd.concat([X[col].astype(str), test[col].astype(str)], axis=0)
        else:
            combined = X[col].astype(str)
        le.fit(combined)
        X[col] = le.transform(X[col].astype(str))
        if col in test.columns:
            test[col] = le.transform(test[col].astype(str))

feature_cols = X.columns.tolist()
print(f'Features: {len(feature_cols)} | Rows: {X.shape[0]}')

Features: 19 | Rows: 630000


In [30]:
sample_weights = compute_sample_weight(class_weight='balanced', y=y)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# stratified used for maintaining class distribution across folds, important for imbalanced datasets

In [31]:
def evaluate_boosting_models(model_name, estimator_class, param_grid):
    results = []

    for params in param_grid:
        fold_scores = []
        for train_idx, valid_idx in skf.split(X, y):
            model = estimator_class(**params, random_state=42, n_jobs=-1)

            if model_name == 'XGBoost':
                model.fit(
                    X.iloc[train_idx], y.iloc[train_idx],
                    sample_weight=sample_weights[train_idx],
                    eval_set=[(X.iloc[valid_idx], y.iloc[valid_idx])],
                    verbose=False,
                )
            else:
                model.fit(
                    X.iloc[train_idx], y.iloc[train_idx],
                    sample_weight=sample_weights[train_idx]
                )

            preds = model.predict(X.iloc[valid_idx])
            fold_scores.append(accuracy_score(y.iloc[valid_idx], preds))

        results.append({
            'model': model_name,
            'params': params,
            'mean_accuracy': np.mean(fold_scores),
            'std_accuracy': np.std(fold_scores),
            'fold_scores': fold_scores,
        })

    return results

In [32]:
xgb_param_grid = [
    {
        'n_estimators': 200,
        'max_depth': 5,
        'learning_rate': 0.10,
        'subsample': 0.80,
        'colsample_bytree': 0.80,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'use_label_encoder': False,
    },
    {
        'n_estimators': 300,
        'max_depth': 7,
        'learning_rate': 0.05,
        'subsample': 0.90,
        'colsample_bytree': 0.70,
        'gamma': 1.0,
        'reg_alpha': 0.5,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'use_label_encoder': False,
    },
    {
        'n_estimators': 150,
        'max_depth': 3,
        'learning_rate': 0.20,
        'subsample': 1.0,
        'colsample_bytree': 0.60,
        'min_child_weight': 3,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',
        'tree_method': 'hist',
        'use_label_encoder': False,
    },
]

lgbm_param_grid = [
    {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'num_leaves': 31,
        'min_child_samples': 20,
        'subsample': 0.80,
        'feature_fraction': 0.80,
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
    },
    {
        'n_estimators': 250,
        'learning_rate': 0.10,
        'num_leaves': 50,
        'min_child_samples': 10,
        'reg_alpha': 1.0,
        'reg_lambda': 1.0,
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
    },
    {
        'n_estimators': 400,
        'learning_rate': 0.03,
        'num_leaves': 16,
        'min_child_samples': 40,
        'feature_fraction': 0.70,
        'bagging_fraction': 0.80,
        'bagging_freq': 1,
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_logloss',
    },
]

In [33]:
xgb_results = evaluate_boosting_models('XGBoost', XGBClassifier, xgb_param_grid)
lgbm_results = evaluate_boosting_models('LightGBM', LGBMClassifier, lgbm_param_grid)

c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:37:58] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:39:16] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:40:27] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:41:38] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encode

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042762 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2696
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 19
[LightGBM] [Info] Start training from score -1.098629
[LightGBM] [Info] Start training from score -1.098628
[LightGBM] [Info] Start training from score -1.098580
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, co

In [34]:
for result in xgb_results + lgbm_results:
    print('Model:', result['model'])
    print('Hyperparameters:')
    for key, value in result['params'].items():
        print(f'  {key}: {value}')
    print(f"Mean Accuracy: {result['mean_accuracy']:.5f}")
    print(f"Std Accuracy: {result['std_accuracy']:.5f}")
    print(f"Fold scores: {[round(s, 5) for s in result['fold_scores']]}")
    print()

Model: XGBoost
Hyperparameters:
  n_estimators: 200
  max_depth: 5
  learning_rate: 0.1
  subsample: 0.8
  colsample_bytree: 0.8
  objective: multi:softprob
  num_class: 3
  eval_metric: mlogloss
  tree_method: hist
  use_label_encoder: False
Mean Accuracy: 0.98283
Std Accuracy: 0.00017
Fold scores: [0.98299, 0.98264, 0.98263, 0.98302, 0.98287]

Model: XGBoost
Hyperparameters:
  n_estimators: 300
  max_depth: 7
  learning_rate: 0.05
  subsample: 0.9
  colsample_bytree: 0.7
  gamma: 1.0
  reg_alpha: 0.5
  objective: multi:softprob
  num_class: 3
  eval_metric: mlogloss
  tree_method: hist
  use_label_encoder: False
Mean Accuracy: 0.98413
Std Accuracy: 0.00016
Fold scores: [0.98429, 0.98407, 0.98393, 0.98435, 0.98402]

Model: XGBoost
Hyperparameters:
  n_estimators: 150
  max_depth: 3
  learning_rate: 0.2
  subsample: 1.0
  colsample_bytree: 0.6
  min_child_weight: 3
  objective: multi:softprob
  num_class: 3
  eval_metric: mlogloss
  tree_method: hist
  use_label_encoder: False
Mean Acc

In [35]:
# Best XGBoost hyperparameters
best_xgb_params = {
    'n_estimators': 200,
    'max_depth': 5,
    'learning_rate': 0.10,
    'subsample': 0.80,
    'colsample_bytree': 0.80,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'tree_method': 'hist',
    'use_label_encoder': False,
    'random_state': 42,
    'n_jobs': -1
}

# Best LightGBM hyperparameters
best_lgbm_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_child_samples': 20,
    'subsample': 0.80,
    'feature_fraction': 0.80,
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'random_state': 42,
    'n_jobs': -1
}

In [36]:
INV_MAP = {0: 'Low', 1: 'Medium', 2: 'High'}

In [37]:
best_xgb = XGBClassifier(**best_xgb_params)
best_xgb.fit(X, y, sample_weight=sample_weights)

# Predict on test
xgb_test_preds = best_xgb.predict(test[feature_cols])
xgb_sub = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': [INV_MAP[p] for p in xgb_test_preds]
})
xgb_sub.to_csv('submission_xgb_best.csv', index=False)

c:\Users\ivpri\gsb545\.venv_clean\Lib\site-packages\xgboost\training.py:200: UserWarning: [13:58:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [38]:
best_lgbm = LGBMClassifier(**best_lgbm_params)
best_lgbm.fit(X, y, sample_weight=sample_weights)

# Predict on test
lgbm_test_preds = best_lgbm.predict(test[feature_cols])
lgbm_sub = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': [INV_MAP[p] for p in lgbm_test_preds]
})
lgbm_sub.to_csv('submission_lgbm_best.csv', index=False)

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.132147 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2701
[LightGBM] [Info] Number of data points in the train set: 630000, number of used features: 19
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
